In [2]:
!pip install duckdb


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import duckdb
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):
  df_var =  "df__" if line is None else line
  globals()[df_var] = duckdb.sql(f"""{cell}""").df()
  return globals()[df_var]

In [4]:
%%sql

select * from df

CatalogException: Catalog Error: Table with name df does not exist!
Did you mean "pg_depend"?

[Dados Abertos Voos e Operações Aereas](https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/)

In [1]:
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/08%20-%20Agosto/VRA_20258.csv"
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/09%20-%20Setembro/VRA_20259.csv"
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/10%20-%20Outubro/VRA_202510.csv"
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/11%20-%20Novembro/VRA_202511.csv"

--2026-05-14 19:52:53--  https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/08%20-%20Agosto/VRA_20258.csv
Resolving sistemas.anac.gov.br (sistemas.anac.gov.br)... 189.84.138.174
Connecting to sistemas.anac.gov.br (sistemas.anac.gov.br)|189.84.138.174|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12039867 (11M) [application/octet-stream]
Saving to: ‘VRA_20258.csv’

VRA_20258.csv       100%[===================>]  11.48M  3.38MB/s    in 3.4s    

2026-05-14 19:52:56 (3.38 MB/s) - ‘VRA_20258.csv’ saved [12039867/12039867]

--2026-05-14 19:52:56--  https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/09%20-%20Setembro/VRA_20259.csv
Resolving sistemas.anac.gov.br (sistemas.anac.gov.br)... 189.84.138.174
Connecting to sistemas.anac.gov.br (sistemas.anac.gov.br)|189.84.138.174|:443... connected.
HTTP request sent, awaiti

In [5]:
VRA_20258 = pd.read_csv('VRA_20258.csv',header='infer',sep=';',skiprows=1, low_memory=False)
VRA_20259 = pd.read_csv('VRA_20259.csv',header='infer',sep=';',skiprows=1, low_memory=False)
VRA_202510 = pd.read_csv('VRA_202510.csv',header='infer',sep=';',skiprows=1, low_memory=False)
VRA_202511 = pd.read_csv('VRA_202511.csv',header='infer',sep=';',skiprows=1, low_memory=False)

VRA = pd.concat([VRA_20258, VRA_20259, VRA_202510, VRA_202511])


VRA = VRA.rename(columns={'ICAO Empresa Aérea': 'icao_empresa_aerea',
                          'Número Voo': 'numero_voo',
                          'Código Autorização (DI)': 'codigo_autorizacao',
                          'Código Tipo Linha': 'codigo_tipo_linha',
                          'ICAO Aeródromo Origem': 'icao_aerodromo_origem',
                          'ICAO Aeródromo Destino': 'icaco_aerodromo_destino',
                          'Partida Prevista': 'partica_prevista',
                          'Partida Real': 'partida_real',
                          'Chegada Prevista': 'chegada_prevista',
                          'Chegada Real': 'chegada_real',
                          'Situação Voo': 'situacao_voo',
                          'Código Justificativa': 'codigo_justificativa',
                          'Justificativa': 'justificativa',
                          'Código Resposta Autorização': 'codigo_resposta_autorizacao',
                          'Resposta Autorização': 'resposta_autorizacao',
                          })

VRA['icao_empresa_aerea'] = VRA['icao_empresa_aerea'].astype(str)
VRA['partida_real'] = pd.to_datetime(VRA['partida_real'], errors='coerce')

VRA['year'] = VRA['partida_real'].dt.year
VRA['month'] = VRA['partida_real'].dt.month
VRA['day'] = VRA['partida_real'].dt.day

VRA['year'] = VRA['year'].fillna(0).astype(int)
VRA['month'] = VRA['month'].fillna(0).astype(int)
VRA['day'] = VRA['day'].fillna(0).astype(int)

VRA.dtypes

icao_empresa_aerea                 object
numero_voo                         object
codigo_autorizacao                 object
codigo_tipo_linha                  object
icao_aerodromo_origem              object
icaco_aerodromo_destino            object
partica_prevista                   object
partida_real               datetime64[ns]
chegada_prevista                   object
chegada_real                       object
situacao_voo                       object
codigo_justificativa              float64
year                                int64
month                               int64
day                                 int64
dtype: object

In [6]:
VRA.to_csv('VRA_full.csv', index=False)

In [7]:
!ls -lh

total 177784
-rw-r--r--@ 1 fabiokishino  staff    38K May 14 19:50 Aula_2_Arquivos_Colunares.ipynb
-rw-r--r--@ 1 fabiokishino  staff    12M Apr 30 10:34 VRA_202510.csv
-rw-r--r--@ 1 fabiokishino  staff    11M Apr 30 10:34 VRA_202511.csv
-rw-r--r--@ 1 fabiokishino  staff    11M Apr 30 10:34 VRA_20258.csv
-rw-r--r--@ 1 fabiokishino  staff    11M Apr 30 10:34 VRA_20259.csv
-rw-r--r--@ 1 fabiokishino  staff    39M May 14 19:55 VRA_full.csv


In [8]:
VRA.to_parquet('VRA_full.parquet', index=False,compression='snappy')

In [9]:
VRA.to_parquet('.', index=False,compression='snappy', partition_cols=['year','month'])

In [10]:
!ls -lhR year\=2025

total 0
drwxr-xr-x@ 3 fabiokishino  staff    96B May 14 19:55 month=10
drwxr-xr-x@ 3 fabiokishino  staff    96B May 14 19:55 month=11
drwxr-xr-x@ 3 fabiokishino  staff    96B May 14 19:55 month=12
drwxr-xr-x@ 3 fabiokishino  staff    96B May 14 19:55 month=7
drwxr-xr-x@ 3 fabiokishino  staff    96B May 14 19:55 month=8
drwxr-xr-x@ 3 fabiokishino  staff    96B May 14 19:55 month=9

year=2025/month=10:
total 3760
-rw-r--r--@ 1 fabiokishino  staff   1.8M May 14 19:55 34aa917ec23f4217959cc70f90cb186d-0.parquet

year=2025/month=11:
total 3688
-rw-r--r--@ 1 fabiokishino  staff   1.8M May 14 19:55 34aa917ec23f4217959cc70f90cb186d-0.parquet

year=2025/month=12:
total 40
-rw-r--r--@ 1 fabiokishino  staff    18K May 14 19:55 34aa917ec23f4217959cc70f90cb186d-0.parquet

year=2025/month=7:
total 24
-rw-r--r--@ 1 fabiokishino  staff    11K May 14 19:55 34aa917ec23f4217959cc70f90cb186d-0.parquet

year=2025/month=8:
total 3696
-rw-r--r--@ 1 fabiokishino  staff   1.8M May 14 19:55 34aa917ec23f4217959cc

In [11]:
%%sql
select * from VRA

,icao_empresa_aerea,numero_voo,codigo_autorizacao,codigo_tipo_linha,icao_aerodromo_origem,icaco_aerodromo_destino,partica_prevista,partida_real,chegada_prevista,chegada_real,situacao_voo,codigo_justificativa,year,month,day
0,TAP,0084,0,I,SBGR,LPPT,2025-08-14 00:45:00,2025-08-14 00:37:00,2025-08-14 10:35:00,2025-08-14 10:41:00,REALIZADO,NaN,2025,8,14
1,TAP,0084,0,I,SBGR,LPPT,2025-08-15 00:40:00,2025-08-15 01:13:00,2025-08-15 10:30:00,2025-08-15 11:04:00,REALIZADO,NaN,2025,8,15
2,TAP,0084,0,I,SBGR,LPPT,2025-08-16 02:25:00,2025-08-16 02:22:00,2025-08-16 12:15:00,2025-08-16 12:04:00,REALIZADO,NaN,2025,8,16
3,TAP,0084,0,I,SBGR,LPPT,2025-08-17 00:45:00,2025-08-17 00:42:00,2025-08-17 10:35:00,2025-08-17 10:41:00,REALIZADO,NaN,2025,8,17
4,TAP,0084,0,I,SBGR,LPPT,2025-08-18 00:45:00,2025-08-18 00:46:00,2025-08-18 10:35:00,2025-08-18 10:51:00,REALIZADO,NaN,2025,8,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335239,TAM,3536,0,N,SBBR,SBRJ,2025-11-07 09:25:00,2025-11-07 09:21:00,2025-11-07 11:05:00,2025-11-07 11:16:00,REALIZADO,NaN,2025,11,7
335240,TAM,3857,0,N,SBSL,SBGR,2025-11-28 20:45:00,NaT,2025-11-29 00:20:00,None,CANCELADO,NaN,0,0,0
335241,TAM,3857,0,N,SBSL,SBGR,2025-11-29 20:45:00,2025-11-29 21:02:00,2025-11-30 00:20:00,2025-11-30 00:32:00,REALIZADO,NaN,2025,11,29
335242,TAM,3857,0,N,SBSL,SBGR,2025-11-30 20:45:00,2025-11-30 20:37:00,2025-12-01 00:20:00,2025-12-01 00:09:00,REALIZADO,NaN,2025,11,30
